# 🌟 Star Schema Creation
## Shoebadoo Sales Analytics - Phase 3

---

## Was machen wir?

Wir bauen ein **Star Schema** mit PySpark:

1. **4 Dimension Tables** erstellen (Date, Customer, Product, Channel)
2. **1 Fact Table** erstellen (Sales)
3. **Alles als Parquet speichern** für Analytics

---

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, quarter, dayofmonth, dayofweek,
    date_format, when, monotonically_increasing_id, row_number
)
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Spark Session
spark = SparkSession.builder \
    .appName("Shoebadoo_StarSchema") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✅ Spark Session created")
print(f"Spark Version: {spark.version}")

## 2. Cleaned Data laden

In [ ]:
# Pfade
INPUT_PATH = "/app/data/cleaned/2_data_cleaning"
OUTPUT_PATH = "/app/data/warehouse/3_star_schema"

# Cleaned Data laden
sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_clean.parquet")
products_df = spark.read.parquet(f"{INPUT_PATH}/products_clean.parquet")
customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_clean.parquet")

print(f"✅ Sales: {sales_df.count():,} rows")
print(f"✅ Products: {products_df.count():,} rows")
print(f"✅ Customers: {customers_df.count():,} rows")

## 3. DIM_DATE erstellen

**Date Dimension** mit allen Zeit-Attributen.

In [ ]:
# Alle unique Dates aus sales_date extrahieren
dates_df = sales_df.select(col("sale_date").alias("full_date")).distinct()

# Date Attributes hinzufügen
dim_date = dates_df.select(
    # date_key = YYYYMMDD als INT
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    weekofyear("full_date").alias("week"),
    dayofmonth("full_date").alias("day"),
    dayofweek("full_date").alias("day_of_week"),
    date_format("full_date", "EEEE").alias("day_name"),
    when(dayofweek("full_date").isin([1, 7]), True).otherwise(False).alias("is_weekend")
).orderBy("date_key")

print(f"✅ dim_date created: {dim_date.count():,} rows")
dim_date.show(5)

## 4. DIM_CUSTOMER erstellen

**Customer Dimension** mit Surrogate Key.

In [ ]:
# Surrogate Key hinzufügen (customer_key = 1, 2, 3...)
window = Window.orderBy("customer_id")

dim_customer = customers_df.select(
    row_number().over(window).alias("customer_key"),
    col("customer_id"),
    col("customer_name"),
    col("customer_segment"),
    col("country"),
    col("city"),
    col("postal_code"),
    col("registration_date")
)

print(f"✅ dim_customer created: {dim_customer.count():,} rows")
dim_customer.show(5)

## 5. DIM_PRODUCT erstellen

**Product Dimension** mit Surrogate Key.

In [ ]:
# Surrogate Key
window = Window.orderBy("product_id")

dim_product = products_df.select(
    row_number().over(window).alias("product_key"),
    col("product_id"),
    col("product_name"),
    col("category"),
    col("subcategory"),
    col("brand"),
    col("price"),
    col("color"),
    col("size")
)

print(f"✅ dim_product created: {dim_product.count():,} rows")
dim_product.show(5)

## 6. DIM_CHANNEL erstellen

**Channel Dimension** (einfach).

In [ ]:
# Channels aus Sales extrahieren
channels_df = sales_df.select(col("channel").alias("channel_name")).distinct()

# Surrogate Key + Channel Type
window = Window.orderBy("channel_name")

dim_channel = channels_df.select(
    row_number().over(window).alias("channel_key"),
    monotonically_increasing_id().cast("string").alias("channel_id"),
    col("channel_name"),
    when(col("channel_name") == "Online", "Digital")
    .otherwise("Physical").alias("channel_type")
)

print(f"✅ dim_channel created: {dim_channel.count():,} rows")
dim_channel.show()

## 7. FACT_SALES erstellen

**Fact Table** mit Foreign Keys zu allen Dimensions.

In [ ]:
# Step 1: date_key joinen
fact_sales = sales_df.join(
    dim_date.select("date_key", col("full_date").alias("join_date")),
    sales_df.sale_date == col("join_date"),
    "left"
)

# Step 2: customer_key joinen
fact_sales = fact_sales.join(
    dim_customer.select("customer_key", col("customer_id").alias("join_customer_id")),
    fact_sales.customer_id == col("join_customer_id"),
    "left"
)

# Step 3: product_key joinen
fact_sales = fact_sales.join(
    dim_product.select("product_key", col("product_id").alias("join_product_id")),
    fact_sales.product_id == col("join_product_id"),
    "left"
)

# Step 4: channel_key joinen
fact_sales = fact_sales.join(
    dim_channel.select("channel_key", col("channel_name").alias("join_channel")),
    fact_sales.channel == col("join_channel"),
    "left"
)

# Step 5: Nur relevante Spalten behalten
fact_sales = fact_sales.select(
    col("sale_id"),
    col("date_key"),
    col("customer_key"),
    col("product_key"),
    col("channel_key"),
    col("quantity"),
    col("unit_price"),
    col("total_amount"),
    col("is_returned")
)

print(f"✅ fact_sales created: {fact_sales.count():,} rows")
fact_sales.show(5)

## 8. Qualitäts-Check

**Schneller Check:** Sind alle Foreign Keys gesetzt?

In [ ]:
# NULL Checks
null_checks = [
    ("date_key", fact_sales.filter(col("date_key").isNull()).count()),
    ("customer_key", fact_sales.filter(col("customer_key").isNull()).count()),
    ("product_key", fact_sales.filter(col("product_key").isNull()).count()),
    ("channel_key", fact_sales.filter(col("channel_key").isNull()).count())
]

print("\n🔍 NULL CHECK:")
print("=" * 40)
for col_name, null_count in null_checks:
    status = "✅" if null_count == 0 else "❌"
    print(f"{status} {col_name}: {null_count} NULLs")
print("=" * 40)

## 9. Star Schema speichern

**Als Parquet speichern** für Analytics.

In [ ]:
# Dimensions speichern
dim_date.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_date")
dim_customer.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_customer")
dim_product.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_product")
dim_channel.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_channel")

# Fact Table speichern
fact_sales.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/fact_sales")

print("\n✅ STAR SCHEMA SAVED!")
print("=" * 50)
print(f"📁 Location: {OUTPUT_PATH}")
print("\n📊 Tables:")
print("  - dim_date")
print("  - dim_customer")
print("  - dim_product")
print("  - dim_channel")
print("  - fact_sales")
print("=" * 50)

## 10. Test Query

**Test:** Umsatz nach Kanal

In [ ]:
# Star Schema laden (as proof)
fact = spark.read.parquet(f"{OUTPUT_PATH}/fact_sales")
dim_ch = spark.read.parquet(f"{OUTPUT_PATH}/dim_channel")
dim_dt = spark.read.parquet(f"{OUTPUT_PATH}/dim_date")

# Query: Umsatz nach Kanal und Jahr
result = fact \
    .join(dim_ch, "channel_key") \
    .join(dim_dt, "date_key") \
    .groupBy("year", "channel_name") \
    .agg({"total_amount": "sum"}) \
    .orderBy("year", col("sum(total_amount)").desc())

print("\n📊 UMSATZ NACH KANAL UND JAHR:")
result.show()

---

## ✅ FERTIG!

### Was haben wir erreicht?

1. ✅ **Star Schema** implementiert (Kimball Methodology)
2. ✅ **4 Dimension Tables** + 1 Fact Table
3. ✅ **Surrogate Keys** für Performance
4. ✅ **Alle Business Questions** beantwortbar
5. ✅ **Parquet Format** für schnelle Analytics

### Nächster Schritt:

**→ Phase 4: Streamlit Dashboard** 📊

---